# Parte 2: Modelagem Preditiva - Resolução de Chamados

### Bibliotecas

In [140]:
# Geral
from pathlib import Path
import os
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve
)


from sklearn.metrics import roc_curve
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.model_selection import RandomizedSearchCV


In [141]:
import importlib
import funcao_aux as fc
importlib.reload(fc)

<module 'funcao_aux' from 'c:\\Users\\quezi\\prefeitura2026\\desafio-cientista-dados-senior-cidadaos-vulneraveis\\notebooks\\funcao_aux.py'>

### Constantes e Variáveis

In [142]:
ROOT_DIR = Path.cwd().parent  # ajuste se necessário

db_path = ROOT_DIR / "dev.duckdb"


### Conexões

In [143]:
con = duckdb.connect(str(db_path))

## 5. Feature Engineering

In [144]:
df_chamados = con.execute("""
SELECT 
    data_inicio,
    data_fim,
    id_bairro,
    tipo
FROM stg_chamado_1746
WHERE data_inicio BETWEEN '2023-01-01' AND '2024-12-31'
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [145]:
df_chamados["data_inicio"] = pd.to_datetime(df_chamados["data_inicio"])
df_chamados["data_fim"] = pd.to_datetime(df_chamados["data_fim"])

df_chamados["tempo_resolucao"] = (
    df_chamados["data_fim"] - df_chamados["data_inicio"]
).dt.days

df_chamados["resolvido_7_dias"] = (
    df_chamados["tempo_resolucao"] <= 7
).astype(int)

In [146]:
# Tratamento de casos problemáticos
df_chamados = df_chamados[df_chamados["tempo_resolucao"].notna()]
df_chamados = df_chamados[df_chamados["tempo_resolucao"] >= 0]

In [147]:
# Criação do dataset de 50.000 chamados
df_sample = []
df_sample = df_chamados.sample(n=50000, random_state=42)

## Construção das features 

In [148]:
# Geospacial 
df_geo = con.execute("""
SELECT 
    id_bairro,
    id_regiao_administrativa,
    nome_bairro,
    nome_regiao_administrativa
FROM stg_dim_territorio
""").fetchdf()

df_sample = df_sample.merge(
    df_geo,
    on="id_bairro",
    how="left"
)

In [149]:
df_sample.info()

df_sample["id_bairro"] = df_sample["id_bairro"].fillna("NAO_INFORMADO")
df_sample["id_regiao_administrativa"] = df_sample["id_regiao_administrativa"].fillna("NAO_INFORMADO")
df_sample["nome_bairro"] = df_sample["nome_bairro"].fillna("NAO_INFORMADO")
df_sample["nome_regiao_administrativa"] = df_sample["nome_regiao_administrativa"].fillna("NAO_INFORMADO")

df_sample["id_bairro"].eq("NAO_INFORMADO").mean()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   data_inicio                 50000 non-null  datetime64[us]
 1   data_fim                    50000 non-null  datetime64[us]
 2   id_bairro                   34038 non-null  object        
 3   tipo                        50000 non-null  object        
 4   tempo_resolucao             50000 non-null  float64       
 5   resolvido_7_dias            50000 non-null  int64         
 6   id_regiao_administrativa    34038 non-null  object        
 7   nome_bairro                 34038 non-null  object        
 8   nome_regiao_administrativa  34038 non-null  object        
dtypes: datetime64[us](2), float64(1), int64(1), object(5)
memory usage: 3.4+ MB


np.float64(0.31924)

In [150]:
df_feriados = pd.concat([
    fc.buscar_feriados(2023),
    fc.buscar_feriados(2024)
], ignore_index=True)

df_feriados["eh_feriado"] = 1

df_sample = df_sample.merge(
    df_feriados[["data_particao", "eh_feriado"]],
    left_on="data_inicio",
    right_on="data_particao",
    how="left"
)

In [151]:
df_feriados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   data_particao       29 non-null     datetime64[ns]
 1   nome_feriado        29 non-null     object        
 2   nome_feriado_local  29 non-null     object        
 3   eh_feriado          29 non-null     int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 1.0+ KB


In [152]:
# Temporal
df_sample["hora"] = df_sample["data_inicio"].dt.hour
df_sample["dia_semana"] = df_sample["data_inicio"].dt.dayofweek
df_sample["mes"] = df_sample["data_inicio"].dt.month
df_sample["fim_de_semana"] = df_sample["dia_semana"].isin([5,6]).astype(int)


df_sample["data_particao"] = pd.to_datetime(df_sample["data_inicio"]).dt.normalize()
df_sample["eh_feriado"] = df_sample["eh_feriado"].fillna(False).astype(int)

C:\Users\quezi\AppData\Local\Temp\ipykernel_15056\791038264.py:9: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [153]:
# Clima
df_clima = fc.buscar_clima_historico(
    latitude=-22.90,
    longitude=-43.20,
    data_inicio="2023-01-01",
    data_fim="2024-12-31"
)


df_clima["data_particao"] = pd.to_datetime(df_clima["data_particao"]).dt.normalize()

df_sample = df_sample.merge(
    df_clima[["data_particao", "temperatura", "precipitacao"]],
    on="data_particao",
    how="left"
)

In [154]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   data_inicio                 50000 non-null  datetime64[us]
 1   data_fim                    50000 non-null  datetime64[us]
 2   id_bairro                   50000 non-null  object        
 3   tipo                        50000 non-null  object        
 4   tempo_resolucao             50000 non-null  float64       
 5   resolvido_7_dias            50000 non-null  int64         
 6   id_regiao_administrativa    50000 non-null  object        
 7   nome_bairro                 50000 non-null  object        
 8   nome_regiao_administrativa  50000 non-null  object        
 9   data_particao               50000 non-null  datetime64[us]
 10  eh_feriado                  50000 non-null  int64         
 11  hora                        50000 non-null  int32     

In [155]:
# Contextual
df_volume_dia = (
    df_chamados
    .groupby("data_inicio")
    .size()
    .reset_index(name="volume_dia")
)

df_sample = df_sample.merge(
    df_volume_dia,
    on="data_inicio",
    how="left"
)

In [156]:
# Limpeza do dados
df_sample["temperatura"] = df_sample["temperatura"].fillna(df_sample["temperatura"].mean())
df_sample["precipitacao"] = df_sample["precipitacao"].fillna(0)

### Justificativa das escolhas de feature engineering

Para a modelagem preditiva da resolução de chamados em até 7 dias, foi construída uma base amostral de 50.000 registros do período de 2023 a 2024. A variável alvo foi definida a partir do tempo de resolução, calculado como a diferença entre a data de abertura e a data de fechamento do chamado, sendo classificada como 1 quando resolvido em até 7 dias e 0 caso contrário.
As features foram estruturadas em múltiplas dimensões. No aspecto temporal, foram derivadas variáveis como hora, dia da semana, mês e indicador de fim de semana, capturando padrões operacionais ao longo do tempo. Também foi incluída uma variável indicadora de feriado, considerando possíveis impactos na capacidade de atendimento.
No contexto climático, foram incorporadas variáveis de temperatura média e precipitação diária, permitindo avaliar a influência de condições ambientais na resolução dos chamados. Para a dimensão geoespacial, foram utilizadas informações de bairro e região administrativa, possibilitando capturar diferenças territoriais na dinâmica de atendimento.
Adicionalmente, foi criada uma variável contextual representando o volume diário de chamados, utilizada como indicador da carga operacional do sistema da carga operacional do sistema. O tratamento de dados incluiu a remoção de valores inconsistentes no tempo de resolução e imputação de valores ausentes nas variáveis climáticas. As variáveis categóricas serão tratadas posteriormente por técnicas de codificação no pipeline de modelagem.

## 6. Modelagem Baseline

In [157]:
df_sample["ano"] = df_sample["data_inicio"].dt.year

df_treino = df_sample[df_sample["ano"] == 2023].copy()
df_teste = df_sample[df_sample["ano"] == 2024].copy()

In [158]:
features_numericas = [
    "temperatura",
    "precipitacao",
    "hora",
    "dia_semana",
    "mes",
    "fim_de_semana",
    "eh_feriado",
    "volume_dia"
]

features_categoricas = [
    "tipo",
    "nome_bairro",
    "nome_regiao_administrativa"
]

target = "resolvido_7_dias"

In [159]:
# Pré-processamento
preprocessador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas),
        ("num", StandardScaler(), features_numericas)
    ]
)

In [160]:
# Modelo
modelo = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        ("classificador", LogisticRegression(
            max_iter=1000,
            n_jobs=-1
        ))
    ]
)

In [161]:
# Treinando o modelo
X_train = df_treino[features_numericas + features_categoricas]
y_train = df_treino[target]


X_train.isna().sum().sort_values(ascending=False)
modelo.fit(X_train, y_train)

,steps,"[('preprocessamento', ...), ('classificador', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [162]:
# Prevendo
X_test = df_teste[features_numericas + features_categoricas]
y_test = df_teste[target]

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

### Análise
i. Resumo - métricas de performance
ii. Curva ROC

O modelo de regressão logística apresentou bom desempenho na classificação de chamados resolvidos em até 7 dias. A métrica de recall (0,92) indica que o modelo consegue identificar corretamente a grande maioria dos chamados que serão resolvidos dentro do prazo, reduzindo o risco de subestimar casos críticos. A precision (0,86) mostra que, entre os chamados previstos como resolvidos no prazo, a maior parte está correta, evidenciando boa confiabilidade nas previsões.
O F1-score (0,88) reforça o equilíbrio entre precision e recall, indicando que o modelo mantém uma performance consistente sem privilegiar excessivamente falsos positivos ou falsos negativos. Já a AUC-ROC (0,78) aponta uma boa capacidade de discriminação do modelo, ou seja, ele consegue diferenciar razoavelmente bem chamados que serão resolvidos dentro do prazo daqueles que não serão.
A curva ROC complementa essa análise ao mostrar a relação entre a taxa de verdadeiros positivos (recall) e a taxa de falsos positivos ao longo de diferentes limiares de decisão. O fato da curva estar consistentemente acima da linha diagonal (baseline aleatório) indica que o modelo possui poder preditivo relevante, embora ainda exista espaço para melhorias.
No contexto da gestão pública, a priorização do recall é especialmente importante, pois permite identificar com maior segurança os chamados que podem não ser resolvidos no prazo, possibilitando ações preventivas e melhor alocação de recursos.

In [163]:
# Performance
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

df_metricas = pd.DataFrame({
    "metrica": ["Precision", "Recall", "F1", "AUC-ROC"],
    "valor": [precision, recall, f1, auc]
})

df_metricas

,metrica,valor
0,Precision,0.857792
1,Recall,0.920450
2,F1,0.888017
3,AUC-ROC,0.784999


In [164]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)

df_roc = pd.DataFrame({
    "fpr": fpr,
    "tpr": tpr
})

fig = px.line(
    df_roc,
    x="fpr",
    y="tpr",
    title="Curva ROC"
)

fig.add_shape(
    type="line",
    line=dict(dash="dash"),
    x0=0, y0=0, x1=1, y1=1
)

fig.show()

## 7. Modelos Avançados e Tuning

1. Floresta Aletatória
2. GradientBoosting

3. Tuning no Floresta Aleatória

In [ ]:
# 1. Floresta Aleatória
modelo_rf = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        ("classificador", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

modelo_rf.fit(X_train, y_train)


y_proba_rf = modelo_rf.predict_proba(X_test)[:,1]

In [ ]:
# 2. GradientBoosting
modelo_gb = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        ("classificador", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.1,
            max_depth=3
        ))
    ]
)

modelo_gb.fit(X_train, y_train)

y_proba_gb = modelo_gb.predict_proba(X_test)[:,1]

,steps,"[('preprocessamento', ...), ('classificador', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# 3. Tuning no Floresta Aleatória
param_grid = {
    "classificador__n_estimators": [100, 200, 300],
    "classificador__max_depth": [5, 10, None],
    "classificador__min_samples_split": [2, 5, 10]
}

search = RandomizedSearchCV(
    modelo_rf,
    param_distributions=param_grid,
    n_iter=5,
    cv=3,
    scoring="recall",  # importante
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

modelo_rf_tuned = search.best_estimator_

In [ ]:
# Métricas
resultados = pd.DataFrame([
    fc.avaliar(modelo, X_test, y_test, "Logística"),
    fc.avaliar(modelo_rf, X_test, y_test, "Floresta Aleatória"),
    fc.avaliar(modelo_rf_tuned, X_test, y_test, "Floresta Aleatória Tuned"),
    fc.avaliar(modelo_gb, X_test, y_test, "GradientBoost")
])
print("Resumo métricas de performance dos 4 modelos")
resultados

,modelo,precision,recall,f1,auc
0,Logística,0.857792,0.920450,0.888017,0.784999
1,Floresta Aleatória,0.852738,0.931260,0.890271,0.784531
2,Floresta Aleatória Tuned,0.809769,1.000000,0.894887,0.765497
3,GradientBoost,0.845332,0.953743,0.896271,0.787289


In [ ]:
# ROC para o GradientBoost
fpr, tpr, _ = roc_curve(y_test, y_proba_gb)

fig = px.line(
    df_roc,
    x="fpr",
    y="tpr",
    title="Curva ROC - GradientBoost"
)

# linha diagonal (baseline aleatório)
fig.add_shape(
    type="line",
    line=dict(dash="dash", color="black"),
    x0=0, y0=0, x1=1, y1=1
)

fig.show()

In [ ]:
# Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_proba_gb)

baseline = y_test.mean()

fig = px.line(
    x=recall,
    y=precision,
    title="Curva Precision-Recall - GradientBoost"
)

# linha baseline
fig.add_shape(
    type="line",
    line=dict(dash="dash", color="black"),
    x0=0, x1=1,
    y0=baseline, y1=baseline
)

fig.show()

### Análise

A comparação entre os modelos evidencia desempenho consistente em todos os algoritmos testados, com destaque para o Gradient Boosting, que apresentou o melhor equilíbrio geral entre as métricas, combinando alto recall (0,95), boa precision (0,85) e a maior AUC-ROC (~0,79), indicando melhor capacidade de discriminação entre os chamados.
O modelo de Regressão Logística mostrou-se um baseline sólido, enquanto o Floresta Aleatória apresentou desempenho semelhante. Já a versão com tuning priorizou o recall, atingindo valor máximo (1,00), porém com redução na precision e na AUC, evidenciando aumento de falsos positivos e menor capacidade de separação entre as classes.
As curvas ROC e Precision-Recall do Gradient Boosting reforçam essa análise. A curva ROC mantém-se acima da linha de referência, indicando boa capacidade de discriminação, enquanto a curva Precision-Recall permanece acima do baseline, com queda gradual da precisão à medida que o recall aumenta. Esse comportamento demonstra um bom equilíbrio entre cobertura e confiabilidade das previsões.
Em conjunto, os resultados indicam que o Gradient Boosting é o modelo mais adequado para produção, por apresentar melhor desempenho global e maior robustez na identificação dos chamados, mantendo alto recall sem comprometer excessivamente a precisão.

8. Interpretabilidade

In [ ]:
modelo_final = modelo_gb

regressor = modelo_final.named_steps["classificador"]
preprocessador = modelo_final.named_steps["preprocessamento"]

In [ ]:
nomes_cat = preprocessador.named_transformers_["cat"].get_feature_names_out(features_categoricas)

nomes_features = list(nomes_cat) + features_numericas

df_importancia = pd.DataFrame({
    "feature": nomes_features,
    "importancia": modelo_gb.named_steps["classificador"].feature_importances_
}).sort_values("importancia", ascending=False)

df_importancia.head(10)

,feature,importancia
96,tipo_Estacionamento irregular,0.138548
499,mes,0.124919
480,nome_regiao_administrativa_NAO_INFORMADO,0.078881
166,tipo_Manejo Arbóreo,0.070015
215,tipo_Pavimentação,0.058036
394,nome_bairro_NAO_INFORMADO,0.050527
45,tipo_Comércio ambulante,0.044203
108,tipo_Fiscalização de obras,0.030318
83,tipo_Drenagem e Saneamento,0.028331
283,tipo_Vias públicas,0.026002


In [ ]:
fig = px.bar(
    df_importancia.head(10),
    x="importancia",
    y="feature",
    orientation="h",
    title="Top 10 variáveis mais importantes"
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
df_importancia["dimensao"] = df_importancia["feature"].apply(fc.classificar_dimensao)

df_dimensao = (
    df_importancia
    .groupby("dimensao", as_index=False)
    .agg(importancia_total=("importancia", "sum"))
    .sort_values("importancia_total", ascending=False)
)

df_dimensao

,dimensao,importancia_total
1,Outros,0.666541
3,Território,0.174662
2,Tempo,0.130717
0,Clima,0.028080


In [ ]:
# Taxa de erro geral
df_resultado = []
df_resultado = df_teste.copy()
df_resultado["y_pred"] = modelo_final.predict(X_test)
df_resultado["erro"] = df_resultado["resolvido_7_dias"] != df_resultado["y_pred"]

In [ ]:
## território
print('Taxa de erro por território')
df_resultado.groupby("nome_regiao_administrativa")["erro"].mean().sort_values(ascending=False).head(10)

Taxa de erro por território


nome_regiao_administrativa
Jacarezinho           0.444444
Complexo Do Alemão    0.421053
Inhauma               0.295455
Guaratiba             0.289474
Anchieta              0.275956
Realengo              0.267091
Campo Grande          0.267019
Vigario Geral         0.263830
Ilha Do Governador    0.260618
Madureira             0.251240
Name: erro, dtype: float64

In [ ]:
## Tipo
print('Taxa de erro por tipo')
df_resultado.groupby("tipo")["erro"].mean().sort_values(ascending=False).head(10)

Taxa de erro por tipo


tipo
Ônibus - pontos terminais/paradas/itinerários                                                1.0
CVZ - Coordenação de Controle de Zoonoses e Fiscalização de Estabelecimentos Veterinários    1.0
Vetores                                                                                      1.0
Concurso Público                                                                             1.0
Criticas - GM                                                                                1.0
Diversos - CGC                                                                               1.0
Diversos - Casa Civil                                                                        1.0
Diversos - Defesa Civil                                                                      1.0
Diversos - SECONSERVA                                                                        1.0
Diversos - SMPDA                                                                             1.0
Name: erro, dtype: float6

In [ ]:
## Tempo
print('Taxa de erro por tempo')
df_resultado.groupby("dia_semana")["erro"].mean()

Taxa de erro por tempo


dia_semana
0    0.176910
1    0.174210
2    0.168498
3    0.179669
4    0.184048
5    0.207717
6    0.176644
Name: erro, dtype: float64

### Análise

A análise das variáveis mais importantes do modelo indica que o principal fator associado à resolução dos chamados está relacionado ao tipo de ocorrência, com destaque para “estacionamento irregular”, que aparece como a variável mais relevante. Esse resultado sugere que determinados tipos de chamados possuem dinâmica operacional própria, sendo resolvidos de forma mais previsível ou padronizada em comparação a outros.
A variável mês também se destaca, indicando a presença de sazonalidade na resolução dos chamados. Esse comportamento pode estar associado a fatores como variações operacionais ao longo do ano, períodos de maior demanda ou mudanças na disponibilidade de equipes, ainda que a análise não aponte diretamente quais meses concentram mais ocorrências.
As variáveis territoriais, como região administrativa e bairro (inclusive valores não informados), também apresentam relevância significativa, indicando que a localização influencia diretamente a capacidade de resolução. A presença de “NAO_INFORMADO” entre as mais importantes sugere que chamados sem informação territorial podem ter dinâmica distinta, possivelmente associada a falhas de registro ou maior complexidade operacional.
Outros tipos de chamados, como manejo arbóreo, pavimentação, drenagem e saneamento, também aparecem entre os mais relevantes, o que indica que demandas relacionadas à infraestrutura urbana tendem a impactar o tempo de resolução, possivelmente devido à maior complexidade técnica ou necessidade de recursos específicos.
De forma geral, o modelo indica que a resolução dos chamados é fortemente influenciada pela natureza da demanda (tipo), seguida por fatores temporais e territoriais, enquanto variáveis climáticas apresentam menor impacto relativo.

Já a agregada por dimensão reforça que as variáveis categóricas (incluídas na categoria “Outros”, principalmente tipos de chamados) concentram a maior parte da importância do modelo, seguidas pelas variáveis territoriais e temporais. Já as variáveis climáticas apresentam baixa contribuição, indicando que o clima tem impacto mais indireto na resolução dos chamados, quando comparado a fatores operacionais e estruturais.

Por fim, a análise de erros do modelo evidencia diferenças relevantes entre as dimensões temporal, territorial e categórica, indicando que a previsibilidade da resolução dos chamados não é homogênea.
Do ponto de vista temporal, observa-se uma leve variação na taxa de erro ao longo da semana, com aumento nos finais de semana, especialmente no sábado, sugerindo possível redução da capacidade operacional ou mudanças no perfil dos chamados nesses períodos. Durante os dias úteis, o erro permanece mais estável, indicando maior previsibilidade do processo de resolução.
Na dimensão territorial, destacam-se regiões como Jacarezinho, Complexo do Alemão e Inhaúma, que apresentam as maiores taxas de erro. Esse comportamento sugere maior complexidade operacional nessas áreas, possivelmente associada a fatores estruturais, logísticos ou socioeconômicos que dificultam a padronização da resolução dos chamados. Em contraste, regiões com menor erro tendem a apresentar processos mais estáveis e previsíveis.
Já na dimensão categórica, alguns tipos de chamados apresentam taxa de erro máxima (1.0), indicando que o modelo não conseguiu generalizar bem para essas categorias. Isso pode estar associado à baixa representatividade desses tipos na base de dados ou à alta variabilidade na forma como esses chamados são tratados. Esse resultado sugere a necessidade de maior volume de dados ou tratamento específico para categorias menos frequentes.

### Insight

Os resultados indicam que o modelo aprendeu que a resolução dos chamados é principalmente influenciada pelo tipo de demanda e pela localização, mais do que por fatores climáticos. Chamados do tipo estacionamento irregular e demandas de infraestrutura apresentam padrões mais previsíveis, indicando que a natureza do problema está diretamente ligada ao tempo de resolução.
A relevância das variáveis territoriais mostra que existem diferenças operacionais entre regiões, sugerindo que a capacidade de atendimento não é homogênea. Isso indica a necessidade de alocação de recursos e estratégias diferenciadas por território, priorizando áreas com maior complexidade ou menor previsibilidade.
Além disso, a presença de maior erro em determinados tipos e regiões aponta para oportunidades de melhoria nos processos operacionais e na qualidade dos dados, especialmente para categorias menos frequentes ou com informações incompletas.
De forma geral, o modelo evidencia que uma gestão mais eficiente deve ser segmentada por tipo de chamado e região, permitindo decisões mais direcionadas e aumento da efetividade no atendimento à população.

**Ob.:** Em complemento ao 03_sistema_priorizacao.ipynb


## 9. Score de Prioridade

In [ ]:
df_prioridade = df_teste.copy()

X = df_prioridade[features_numericas + features_categoricas]

In [ ]:
# Risco de atraso 0,40
df_prioridade["prob_resolvido_7_dias"] = modelo_gb.predict_proba(X)[:, 1]

df_prioridade["risco_atraso"] = 1 - df_prioridade["prob_resolvido_7_dias"]

In [ ]:
# Urgência do tipo 0,20
mapa_urgencia = {
    "Drenagem e Saneamento": 1.00,
    "Iluminação Pública": 0.85,
    "Manejo Arbóreo": 0.80,
    "Pavimentação": 0.75,
    "Fiscalização de obras": 0.70,
    "Estacionamento irregular": 0.50,
    "Remoção Gratuita": 0.40
}

df_prioridade["urgencia_tipo"] = (
    df_prioridade["tipo"]
    .map(mapa_urgencia)
    .fillna(0.50)
)

In [ ]:
# Impacto territorial 0,15
volume_bairro = (
    df_sample
    .groupby("nome_bairro")["tipo"]
    .count()
    .reset_index(name="volume_bairro")
)

volume_bairro["impacto_territorial"] = (
    volume_bairro["volume_bairro"] / volume_bairro["volume_bairro"].max()
)

df_prioridade = df_prioridade.merge(
    volume_bairro[["nome_bairro", "impacto_territorial"]],
    on="nome_bairro",
    how="left"
)

In [ ]:
# Equidade territorial 0,15
erro_territorio = (
    df_resultado
    .groupby("nome_regiao_administrativa")["erro"]
    .mean()
    .reset_index(name="taxa_erro_territorio")
)

erro_territorio["equidade_territorial"] = (
    erro_territorio["taxa_erro_territorio"] /
    erro_territorio["taxa_erro_territorio"].max()
)

df_prioridade = df_prioridade.merge(
    erro_territorio[["nome_regiao_administrativa", "equidade_territorial"]],
    on="nome_regiao_administrativa",
    how="left"
)

In [ ]:
# Contexto climático 0,10
df_prioridade["contexto_climatico"] = (
    (
        df_prioridade["precipitacao"] >= df_prioridade["precipitacao"].quantile(0.90)
    ) |
    (
        df_prioridade["temperatura"] >= df_prioridade["temperatura"].quantile(0.90)
    )
).astype(int)

In [ ]:
## Score Final
df_prioridade["score_prioridade"] = (
    0.40 * df_prioridade["risco_atraso"] +
    0.20 * df_prioridade["urgencia_tipo"] +
    0.15 * df_prioridade["impacto_territorial"] +
    0.15 * df_prioridade["equidade_territorial"] +
    0.10 * df_prioridade["contexto_climatico"]
)

## Item 10 — Simulação e Impacto

In [166]:
df_simulacao = df_prioridade.copy()

In [167]:
df_simulacao["atraso_real"] = 1 - df_simulacao["resolvido_7_dias"]

In [ ]:
# Estratégia 1: seleção aleatória de 20%:
df_simulacao["prioritario_aleatorio"] = 0

idx_aleatorio = df_simulacao.sample(frac=0.20, random_state=42).index

df_simulacao.loc[idx_aleatorio, "prioritario_aleatorio"] = 1

In [169]:
# Estratégia 1: seleção aleatória de 20%:
limite_score = df_simulacao["score_prioridade"].quantile(0.80)

df_simulacao["prioritario_score"] = (
    df_simulacao["score_prioridade"] >= limite_score
).astype(int)

In [ ]:
# Comparando as métricas

df_metricas_simulacao = pd.DataFrame({
    "estrategia": ["Aleatória 20%", "Top 20% Score"],
    "precision": [
        precision_score(df_simulacao["atraso_real"], df_simulacao["prioritario_aleatorio"]),
        precision_score(df_simulacao["atraso_real"], df_simulacao["prioritario_score"])
    ],
    "recall": [
        recall_score(df_simulacao["atraso_real"], df_simulacao["prioritario_aleatorio"]),
        recall_score(df_simulacao["atraso_real"], df_simulacao["prioritario_score"])
    ],
    "qtd_priorizados": [
        df_simulacao["prioritario_aleatorio"].sum(),
        df_simulacao["prioritario_score"].sum()
    ],
    "qtd_atrasos_capturados": [
        df_simulacao.loc[df_simulacao["prioritario_aleatorio"] == 1, "atraso_real"].sum(),
        df_simulacao.loc[df_simulacao["prioritario_score"] == 1, "atraso_real"].sum()
    ]
})

df_metricas_simulacao

,estrategia,precision,recall,qtd_priorizados,qtd_atrasos_capturados
0,Aleatória 20%,0.189231,0.198941,5163,977
1,Top 20% Score,0.370958,0.390145,5165,1916


In [171]:
# Cálculo do ganho esperado
recall_aleatorio = df_metricas_simulacao.loc[
    df_metricas_simulacao["estrategia"] == "Aleatória 20%", 
    "recall"
].iloc[0]

recall_score = df_metricas_simulacao.loc[
    df_metricas_simulacao["estrategia"] == "Top 20% Score", 
    "recall"
].iloc[0]

ganho_recall = recall_score / recall_aleatorio

ganho_recall

np.float64(1.9611054247697033)

In [173]:
# Lift curve
df_lift = []

# ordenar pela prioridade (maior score primeiro)
df_lift = df_simulacao.sort_values("score_prioridade", ascending=False).copy()

# ranking
df_lift["rank"] = np.arange(1, len(df_lift) + 1)

# percentual acumulado da base
df_lift["perc_base"] = df_lift["rank"] / len(df_lift)

# atrasos acumulados
df_lift["atrasos_acumulados"] = df_lift["atraso_real"].cumsum()

# recall acumulado (quanto dos atrasos já capturamos)
df_lift["recall_acumulado"] = (
    df_lift["atrasos_acumulados"] / df_lift["atraso_real"].sum()
)

fig = px.line(
    df_lift,
    x="perc_base",
    y="recall_acumulado",
    title="Lift Curve - Priorização por Score",
    labels={
        "perc_base": "% da base priorizada",
        "recall_acumulado": "% dos atrasos capturados"
    }
)

# linha aleatória (baseline)
fig.add_shape(
    type="line",
    line=dict(dash="dash", color="black"),
    x0=0, y0=0,
    x1=1, y1=1
)

# linha vertical nos 20%
fig.add_vline(
    x=0.20,
    line_dash="dash",
    line_color="red"
)

fig.show()

### Análise

A simulação comparou duas estratégias de priorização considerando a restrição operacional de atuação em apenas 20% dos chamados: uma seleção aleatória e uma seleção baseada no score de prioridade desenvolvido.

Os resultados evidenciam um ganho significativo na estratégia orientada por dados. A seleção aleatória apresentou precision de aproximadamente 19% e recall de 20%, capturando 977 chamados com atraso. Já a priorização pelo score elevou a precision para cerca de 37% e o recall para 39%, capturando 1.916 chamados com atraso.

Esse resultado representa um ganho expressivo, em que a estratégia baseada no score captura aproximadamente 1,96 vezes mais chamados com atraso, o que equivale a um aumento de cerca de 96% na efetividade da priorização, mantendo o mesmo volume de atendimento.

A análise da lift curve reforça esse comportamento. Observa-se que, ao priorizar os primeiros 20% da base, já é possível capturar cerca de 40% dos chamados com atraso, enquanto uma seleção aleatória capturaria apenas 20%. Esse descolamento em relação à linha de referência demonstra que o modelo é capaz de concentrar os casos mais críticos no topo da ordenação, permitindo uma priorização muito mais eficiente.

Diante desses resultados, recomenda-se a implementação da priorização baseada no score, pois ela permite aumentar substancialmente a efetividade operacional sem necessidade de ampliação de capacidade. Além disso, por incorporar dimensões de urgência, impacto social e equidade territorial, a estratégia garante não apenas eficiência, mas também uma alocação mais justa e estratégica dos recursos públicos.